In [28]:
import torch
import torchvision

print(torch.__version__)
print(torchvision.__version__)

2.7.1+cu118
0.22.1+cu118


In [29]:
device="cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

In [30]:
def set_seeds(seed:int=42):
    """sets random seed for torch operation
    Args:
        seed(int,optional):random seed to set. Deafualt to 42.
    """
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)

In [31]:
set_seeds()


In [32]:
#fuction for downloading the data
import os 
import zipfile
from pathlib import Path
import requests

def download_data(source:str,
                  destination:str,
                  remove_sourse:bool=True)->Path:
    """downloads a zipped datasets from source and unzips to destination."""

    # Setup path to data folder
    data_path = Path("data/")
    image_path = data_path / destination

    # If the image folder doesn't exist, download it and prepare it... 
    if image_path.is_dir():
        print(f"{image_path} directory exists,skipping redownloading....")
    else:
        print(f"Did not find {image_path} directory, creating one...")
        image_path.mkdir(parents=True, exist_ok=True)
    
        # Download pizza, steak, sushi data
        target_file=Path(source).name
        with open(data_path /target_file, "wb") as f:
            request = requests.get(source)
            print("Downloading target file from source")
            f.write(request.content)

        # Unzip pizza, steak, sushi data
        with zipfile.ZipFile(data_path / target_file, "r") as zip_ref:
            print(f"Unzipping{target_file} data...") 
            zip_ref.extractall(image_path)
    
        # Remove zip file
        os.remove(data_path /target_file)
    return image_path

In [33]:
image_path=download_data(source="https://github.com/mrdbourke/pytorch-deep-learning/raw/main/data/pizza_steak_sushi.zip",
                         destination="pizza_steak_sushi")

image_path

data\pizza_steak_sushi directory exists,skipping redownloading....


WindowsPath('data/pizza_steak_sushi')

In [34]:
train_dir=image_path/"train"
test_dir=image_path/"test"

In [35]:
#setup imagenet normalization levels
from torchvision import transforms
normalize=transforms.Normalize(mean=[0.485,0.456,0.406],
                               std=[0.229,0.224,0.225])
#transform pipline manually

manual_transforms=transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    normalize
])

print(f"manually created transforms:{manual_transforms}")

from going_modular import data_setup
train_dataloader,test_dataloader,class_names=data_setup.create_dataloaders(
    train_dir=train_dir,
    test_dir=test_dir,
    transform=manual_transforms,
    batch_size=32
)
train_dataloader,test_dataloader,class_names

manually created transforms:Compose(
    Resize(size=(224, 224), interpolation=bilinear, max_size=None, antialias=True)
    ToTensor()
    Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
)


(<torch.utils.data.dataloader.DataLoader at 0x162f4eb6900>,
 ['pizza', 'steak', 'sushi'])

In [36]:
#transform pipeline automatically
import torchvision
weights=torchvision.models.EfficientNet_B0_Weights.DEFAULT
automatic_transforms=weights.transforms()
print(f"automatically created transforms: {automatic_transforms}")

train_dataloader,test_dataloader,class_names=data_setup.create_dataloaders(
    train_dir=train_dir,
    test_dir=test_dir,
    transform=automatic_transforms,
    batch_size=32
)
train_dataloader,test_dataloader,class_names

automatically created transforms: ImageClassification(
    crop_size=[224]
    resize_size=[256]
    mean=[0.485, 0.456, 0.406]
    std=[0.229, 0.224, 0.225]
    interpolation=InterpolationMode.BICUBIC
)


(<torch.utils.data.dataloader.DataLoader at 0x162f9f23860>,
 ['pizza', 'steak', 'sushi'])

In [37]:
# model=torchvision.models.efficientnet_b0(pretrained=True).to(device)
# model

#or

weights=torchvision.models.EfficientNet_B0_Weights.DEFAULT

model=torchvision.models.efficientnet_b0(weights=weights).to(device)
model

EfficientNet(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): SiLU(inplace=True)
    )
    (1): Sequential(
      (0): MBConv(
        (block): Sequential(
          (0): Conv2dNormActivation(
            (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
            (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
            (2): SiLU(inplace=True)
          )
          (1): SqueezeExcitation(
            (avgpool): AdaptiveAvgPool2d(output_size=1)
            (fc1): Conv2d(32, 8, kernel_size=(1, 1), stride=(1, 1))
            (fc2): Conv2d(8, 32, kernel_size=(1, 1), stride=(1, 1))
            (activation): SiLU(inplace=True)
            (scale_activation): Sigmoid()
          )
          (2): Conv2dNormActivat

In [38]:
#to freeze all base layer(feature extraction)
for param in model.features.parameters():
    # print(param)
    param.requires_grad=False

In [39]:
model.classifier

Sequential(
  (0): Dropout(p=0.2, inplace=True)
  (1): Linear(in_features=1280, out_features=1000, bias=True)
)

In [40]:
#adjust classifier output
from torch import nn
set_seeds()
model.classifier=nn.Sequential(
    nn.Dropout(p=0.2,inplace=True),
    nn.Linear(in_features=1280,out_features=len(class_names)).to(device)
)

In [41]:
from torchinfo import summary

summary(model,
        input_size=(32,3,224,224),
        verbose=0,
        col_names=["input_size","output_size","num_params","trainable"],
        col_width=20,
        row_settings=["var_names"])

Layer (type (var_name))                                      Input Shape          Output Shape         Param #              Trainable
EfficientNet (EfficientNet)                                  [32, 3, 224, 224]    [32, 3]              --                   Partial
├─Sequential (features)                                      [32, 3, 224, 224]    [32, 1280, 7, 7]     --                   False
│    └─Conv2dNormActivation (0)                              [32, 3, 224, 224]    [32, 32, 112, 112]   --                   False
│    │    └─Conv2d (0)                                       [32, 3, 224, 224]    [32, 32, 112, 112]   (864)                False
│    │    └─BatchNorm2d (1)                                  [32, 32, 112, 112]   [32, 32, 112, 112]   (64)                 False
│    │    └─SiLU (2)                                         [32, 32, 112, 112]   [32, 32, 112, 112]   --                   --
│    └─Sequential (1)                                        [32, 32, 112, 112]   [32, 

In [42]:
loss_fn=nn.CrossEntropyLoss()
optimizer=torch.optim.Adam(model.parameters(),lr=0.001)

In [43]:
#setup a summarywriter

from torch.utils.tensorboard import SummaryWriter
writer=SummaryWriter()
writer

In [44]:
from going_modular.engine import train_step,test_step
from typing import Dict,List,Tuple
from tqdm.auto import tqdm

def train(model: torch.nn.Module, 
          train_dataloader: torch.utils.data.DataLoader, 
          test_dataloader: torch.utils.data.DataLoader, 
          optimizer: torch.optim.Optimizer,
          loss_fn: torch.nn.Module,
          epochs: int,
          device: torch.device) -> Dict[str, List[float]]:
  """Trains and tests a PyTorch model.

  Passes a target PyTorch models through train_step() and test_step()
  functions for a number of epochs, training and testing the model
  in the same epoch loop.

  Calculates, prints and stores evaluation metrics throughout.

  Args:
    model: A PyTorch model to be trained and tested.
    train_dataloader: A DataLoader instance for the model to be trained on.
    test_dataloader: A DataLoader instance for the model to be tested on.
    optimizer: A PyTorch optimizer to help minimize the loss function.
    loss_fn: A PyTorch loss function to calculate loss on both datasets.
    epochs: An integer indicating how many epochs to train for.
    device: A target device to compute on (e.g. "cuda" or "cpu").

  Returns:
    A dictionary of training and testing loss as well as training and
    testing accuracy metrics. Each metric has a value in a list for 
    each epoch.
    In the form: {train_loss: [...],
                  train_acc: [...],
                  test_loss: [...],
                  test_acc: [...]} 
    For example if training for epochs=2: 
                 {train_loss: [2.0616, 1.0537],
                  train_acc: [0.3945, 0.3945],
                  test_loss: [1.2641, 1.5706],
                  test_acc: [0.3400, 0.2973]} 
  """
  # Create empty results dictionary
  results = {"train_loss": [],
      "train_acc": [],
      "test_loss": [],
      "test_acc": []
  }

  # Loop through training and testing steps for a number of epochs
  for epoch in tqdm(range(epochs)):
      train_loss, train_acc = train_step(model=model,
                                          dataloader=train_dataloader,
                                          loss_fn=loss_fn,
                                          optimizer=optimizer,
                                          device=device)
      test_loss, test_acc = test_step(model=model,
          dataloader=test_dataloader,
          loss_fn=loss_fn,
          device=device)

      # Print out what's happening
      print(
          f"Epoch: {epoch+1} | "
          f"train_loss: {train_loss:.4f} | "
          f"train_acc: {train_acc:.4f} | "
          f"test_loss: {test_loss:.4f} | "
          f"test_acc: {test_acc:.4f}"
      )

      # Update results dictionary
      results["train_loss"].append(train_loss)
      results["train_acc"].append(train_acc)
      results["test_loss"].append(test_loss)
      results["test_acc"].append(test_acc)

      ##new:experiment trackin
      writer.add_scalars(main_tag="Loss",
                        tag_scalar_dict={"train_loss": train_loss,
                                         "test_loss":test_loss},
                        global_step=epoch)
      writer.add_scalars(main_tag="Accuracy",
                        tag_scalar_dict={"train_acc":train_acc,
                                         "test_acc":test_acc},
                        global_step=epoch)
      writer.add_graph(model=model,
                       input_to_model=torch.rand(32,3,224,224).to(device))

  writer.close()
  # Return the filled results at the end of the epochs
  return results


In [45]:
set_seeds()
results=train(model=model,
              train_dataloader=train_dataloader,
              test_dataloader=test_dataloader,
              optimizer=optimizer,
              loss_fn=loss_fn,
              epochs=5,
              device=device)

  0%|          | 0/5 [00:00<?, ?it/s]

Epoch: 1 | train_loss: 1.0947 | train_acc: 0.4023 | test_loss: 0.9138 | test_acc: 0.5502


 20%|██        | 1/5 [00:12<00:51, 12.77s/it]

Epoch: 2 | train_loss: 0.8952 | train_acc: 0.6562 | test_loss: 0.7852 | test_acc: 0.8258


 40%|████      | 2/5 [00:29<00:45, 15.08s/it]

...

 60%|██████    | 3/5 [00:45<00:31, 15.58s/it]

Epoch: 4 | train_loss: 0.6844 | train_acc: 0.8516 | test_loss: 0.6712 | test_acc: 0.8561


 80%|████████  | 4/5 [00:59<00:14, 14.86s/it]

Epoch: 5 | train_loss: 0.7056 | train_acc: 0.7227 | test_loss: 0.6758 | test_acc: 0.7633


100%|██████████| 5/5 [01:14<00:00, 14.91s/it]


In [46]:
%load_ext tensorboard
%tensorboard --logdir runs

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


Reusing TensorBoard on port 6006 (pid 12788), started 6 days, 10:25:53 ago. (Use '!kill 12788' to kill it.)

In [47]:
#create a function to prepare a summarywriter() instance
from torch.utils.tensorboard import SummaryWriter
def create_writer(experiment_name:str,
                  model_name:str,
                  extra:str=None):
    """creates a torch.utils.tensorboard.writer.summarywriter() instance tracking  to a spcefic directory"""
    from datetime import datetime
    import os
    timestamp=datetime.now().strftime("%Y-%m-%d")
    if extra:
        log_dir=os.path.join("runs",timestamp,experiment_name,model_name,extra)
    else:
        log_dir=os.path.join("runs",timestamp,experiment_name,model_name)
    print(f"[INFO] created summaarywriter saveing to {log_dir}")
    return SummaryWriter(log_dir=log_dir)
    

In [48]:
example_writer=create_writer(experiment_name="data_10__percent",
                             model_name="effnetb0",
                             extra="5_epochs")
example_writer

[INFO] created summaarywriter saveing to runs\2025-07-06\data_10__percent\effnetb0\5_epochs


In [49]:
import torch.utils.tensorboard
from going_modular.engine import train_step,test_step
from typing import Dict,List,Tuple
from tqdm.auto import tqdm

def train(model: torch.nn.Module, 
          train_dataloader: torch.utils.data.DataLoader, 
          test_dataloader: torch.utils.data.DataLoader, 
          optimizer: torch.optim.Optimizer,
          loss_fn: torch.nn.Module,
          epochs: int,
          device: torch.device,
          writer:torch.utils.tensorboard.SummaryWriter) -> Dict[str, List[float]]:
  """Trains and tests a PyTorch model.

  Passes a target PyTorch models through train_step() and test_step()
  functions for a number of epochs, training and testing the model
  in the same epoch loop.

  Calculates, prints and stores evaluation metrics throughout.

  Args:
    model: A PyTorch model to be trained and tested.
    train_dataloader: A DataLoader instance for the model to be trained on.
    test_dataloader: A DataLoader instance for the model to be tested on.
    optimizer: A PyTorch optimizer to help minimize the loss function.
    loss_fn: A PyTorch loss function to calculate loss on both datasets.
    epochs: An integer indicating how many epochs to train for.
    device: A target device to compute on (e.g. "cuda" or "cpu").

  Returns:
    A dictionary of training and testing loss as well as training and
    testing accuracy metrics. Each metric has a value in a list for 
    each epoch.
    In the form: {train_loss: [...],
                  train_acc: [...],
                  test_loss: [...],
                  test_acc: [...]} 
    For example if training for epochs=2: 
                 {train_loss: [2.0616, 1.0537],
                  train_acc: [0.3945, 0.3945],
                  test_loss: [1.2641, 1.5706],
                  test_acc: [0.3400, 0.2973]} 
  """
  # Create empty results dictionary
  results = {"train_loss": [],
      "train_acc": [],
      "test_loss": [],
      "test_acc": []
  }

  # Loop through training and testing steps for a number of epochs
  for epoch in tqdm(range(epochs)):
      train_loss, train_acc = train_step(model=model,
                                          dataloader=train_dataloader,
                                          loss_fn=loss_fn,
                                          optimizer=optimizer,
                                          device=device)
      test_loss, test_acc = test_step(model=model,
          dataloader=test_dataloader,
          loss_fn=loss_fn,
          device=device)

      # Print out what's happening
      print(
          f"Epoch: {epoch+1} | "
          f"train_loss: {train_loss:.4f} | "
          f"train_acc: {train_acc:.4f} | "
          f"test_loss: {test_loss:.4f} | "
          f"test_acc: {test_acc:.4f}"
      )

      # Update results dictionary
      results["train_loss"].append(train_loss)
      results["train_acc"].append(train_acc)
      results["test_loss"].append(test_loss)
      results["test_acc"].append(test_acc)



      ##new:experiment trackin
      if writer:
        writer.add_scalars(main_tag="Loss",
                            tag_scalar_dict={"train_loss": train_loss,
                                            "test_loss":test_loss},
                            global_step=epoch)
        writer.add_scalars(main_tag="Accuracy",
                            tag_scalar_dict={"train_acc":train_acc,
                                            "test_acc":test_acc},
                            global_step=epoch)
        writer.add_graph(model=model,
                        input_to_model=torch.rand(32,3,224,224).to(device))

        writer.close()
      else:
         pass
  # Return the filled results at the end of the epochs
  return results


In [50]:
#now we will experiment and run 2 different model with different hyper parameters with differennt datasets and many more

#downloading different datasets

# Download 10 percent and 20 percent datasets
data_10_percent_path = download_data(source="https://github.com/mrdbourke/pytorch-deep-learning/raw/main/data/pizza_steak_sushi.zip",
                                     destination="pizza_steak_sushi")


data_20_percent_path = download_data(source="https://github.com/mrdbourke/pytorch-deep-learning/raw/main/data/pizza_steak_sushi_20_percent.zip",
                                     destination="pizza_steak_sushi_20_percent")

data\pizza_steak_sushi directory exists,skipping redownloading....
data\pizza_steak_sushi_20_percent directory exists,skipping redownloading....


In [54]:
train_dir_10=data_10_percent_path/"train"
train_dir_20=data_20_percent_path/"train"

test_dir=data_10_percent_path/"test"

train_dir_10,train_dir_20,test_dir

(WindowsPath('data/pizza_steak_sushi/train'),
 WindowsPath('data/pizza_steak_sushi_20_percent/train'),
 WindowsPath('data/pizza_steak_sushi/test'))

In [55]:
normalize=transforms.Normalize(mean=[0.485,0.456,0.406],
                               std=[0.229,0.224,0.225])

simple_transform=transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    normalize
])

In [ ]:
BATCH_SIZE=32

train_dataloader_10,test_dataloader,class_names=data_setup.create_dataloaders(train_dir=train_dir_10,
                                                                              test_dir=test_dir,
                                                                              transform=simple_transform,
                                                                              batch_size=BATCH_SIZE
                                                                              )

train_dataloader_20,test_dataloader,class_names=data_setup.create_dataloaders(train_dir=train_dir_20,
                                                                              test_dir=test_dir,
                                                                              transform=simple_transform,
                                                                              batch_size=BATCH_SIZE
                                                                              )
